# 03. Метод главных компонент на шкале интенсивности (метод 9)

**Единственная ML-часть отчёта.**

Берём три ряда **интенсивности** на уровне страна–год:
1. GERD в процентах ВВП;
2. BERD (PERFORMED) в процентах ВВП;
3. исследователи на миллион жителей.

Стандартизируем столбцы, оцениваем метод главных компонент. Первая главная компонента сжимает сонаправленные ряды; смотрим долю объяснённой дисперсии и нагрузки. Это **не** «индекс технологической мощи» и не рейтинг способностей.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
PANEL_PATH = ROOT / "data_reviewed" / "core_panel_reviewed.csv"
TABLES = ROOT / "data_reviewed" / "tables_reviewed"
RAW = ROOT / "data" / "raw"
OUT_FIG = ROOT / "notebooks" / "figures"
OUT_TAB = ROOT / "notebooks" / "tables"
OUT_FIG.mkdir(parents=True, exist_ok=True)
OUT_TAB.mkdir(parents=True, exist_ok=True)

panel = pd.read_csv(PANEL_PATH)
panel = panel[(panel["year"] >= 2010) & (panel["year"] <= 2024)].copy()
US_CN = panel[panel["country_iso3"].isin(["USA", "CHN"])].copy()

def cagr(start: float, end: float, n_years: int) -> float:
    """Compound annual growth rate; n_years = end_year - start_year."""
    if start is None or end is None or np.isnan(start) or np.isnan(end) or start <= 0 or n_years <= 0:
        return np.nan
    return (end / start) ** (1 / n_years) - 1

print("panel rows", len(panel), "US+CN rows", len(US_CN))
print("countries", sorted(panel["country_iso3"].unique()))

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


panel rows 120 US+CN rows 30
countries ['CHN', 'DEU', 'FRA', 'GBR', 'ISR', 'JPN', 'KOR', 'USA']


In [2]:
cols = ["gerd_pct_gdp", "berd_pct_gdp", "researchers_per_million"]
pca_df = panel[["country_iso3", "year"] + cols].dropna().copy()
print("complete country-year rows:", len(pca_df))
print(pca_df.groupby("country_iso3").size().sort_values(ascending=False))

X = pca_df[cols].to_numpy()
scaler = StandardScaler()
Z = scaler.fit_transform(X)
pca = PCA(n_components=3, random_state=0)
scores = pca.fit_transform(Z)

var = pd.DataFrame({
    "PC": [1, 2, 3],
    "variance_explained": pca.explained_variance_ratio_,
    "variance_explained_pct": 100 * pca.explained_variance_ratio_,
})
loadings = pd.DataFrame(pca.components_.T, index=cols, columns=["PC1", "PC2", "PC3"])
pca_df = pca_df.assign(PC1=scores[:, 0], PC2=scores[:, 1])

var.to_csv(OUT_TAB / "03_pca_variance.csv", index=False)
loadings.to_csv(OUT_TAB / "03_pca_loadings.csv")
pca_df.to_csv(OUT_TAB / "03_pca_scores.csv", index=False)

print("Variance explained:")
print(var.to_string(index=False))
print("Loadings:")
print(loadings.to_string())
var


complete country-year rows: 91
country_iso3
CHN    14
DEU    14
FRA    14
JPN    14
KOR    14
USA    13
GBR     8
dtype: int64
Variance explained:
 PC  variance_explained  variance_explained_pct
  1            0.907450               90.744966
  2            0.088927                8.892653
  3            0.003624                0.362381
Loadings:
                              PC1       PC2       PC3
gerd_pct_gdp             0.597951 -0.277108  0.752107
berd_pct_gdp             0.584859 -0.490786 -0.645809
researchers_per_million  0.548082  0.826039 -0.131397


,PC,variance_explained,variance_explained_pct
0,1,0.907450,90.744966
1,2,0.088927,8.892653
2,3,0.003624,0.362381


In [3]:
# Latest year per country for a simple country scatter on PC1
latest = pca_df.sort_values("year").groupby("country_iso3", as_index=False).tail(1)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(latest["country_iso3"], latest["PC1"], color="#4c72b0")
for _, r in latest.iterrows():
    if r["country_iso3"] in ("USA", "CHN"):
        ax.bar([r["country_iso3"]], [r["PC1"]], color="#d62728" if r["country_iso3"] == "CHN" else "#1f77b4")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_title("Первая главная компонента интенсивности (последний доступный год по стране)")
ax.set_ylabel("PC1 (стандартизованная шкала)")
ax.set_xlabel("Страна")
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_FIG / "03_pca_pc1_by_country.png", dpi=150)
plt.show()
latest[["country_iso3", "year", "PC1", "gerd_pct_gdp", "berd_pct_gdp", "researchers_per_million"]]


C:\Users\danii\AppData\Local\Temp\ipykernel_19104\4061907823.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,country_iso3,year,PC1,gerd_pct_gdp,berd_pct_gdp,researchers_per_million
92,GBR,2017,-0.729990,2.32602,1.780212,4472.69458
197,USA,2022,1.180713,3.48736,2.729052,4937.49441
23,CHN,2023,-1.025353,2.57729,2.002840,2107.27003
48,DEU,2023,0.683107,3.15390,2.142713,5926.07432
148,JPN,2023,1.335717,3.44125,2.723354,5608.63075
73,FRA,2023,-0.889582,2.17832,1.437429,5368.63555
173,KOR,2023,4.733389,4.94352,3.914864,9471.82543


## Краткий вывод для отчёта (метод 9)

Первая компонента объясняет большую долю дисперсии трёх intensity-рядов (ожидаемо порядка ~90%, как в `results/pca_repair_a_intensity_variance.csv`). Нагрузки показывают, что GERD%, BERD% и исследователи на млн движутся в одну сторону. США и Китай различаются по положению на PC1 — это сжатие выбранных шкал интенсивности, не вердикт о «мощи».
